# Onshore wind NPV simulation

Run the onshore wind electricity Monte Carlo simulation and visualize the resulting NPV distribution.

The summary also reports how many simulations have non-negative NPV (NPV >= 0) and how many have negative NPV.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from electricity.electricity_npv_monte_carlo import (
    DEFAULT_RANDOM_SEED,
    DEFAULT_SAMPLE_SIZE,
    simulate_electricity_results,
)

from npv_summary import summarize_metric_signs


In [2]:
TECHNOLOGY = 'wind_onshore'
SAMPLE_SIZE = DEFAULT_SAMPLE_SIZE
RANDOM_SEED = DEFAULT_RANDOM_SEED

results_by_technology = simulate_electricity_results(
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
    technologies=(TECHNOLOGY,),
)
simulation = results_by_technology[TECHNOLOGY]
results = pd.DataFrame(simulation)
results.head()


,run_id,technology,annual_output_mwh,full_load_hours_per_year,lifetime_years,capacity_mw,capacity_kw,capex_eur_per_kw,fixed_opex_eur_per_kw_year,variable_opex_eur_per_mwh,...,annual_variable_opex_eur,annual_fuel_cost_eur,annual_emissions_cost_eur,annual_total_cost_eur,annual_net_cash_flow_eur,npv_eur,discounted_lifetime_output_mwh,present_value_total_cost_eur,lcoe_eur_per_mwh,levelized_net_margin_eur_per_mwh
0,0,wind_onshore,1000000.0,2500.0,25.0,400.0,400000.0,1883.475250,32.945197,6.987695,...,6.987695e+06,0.0,0.0,2.016577e+07,7.390423e+07,3.552097e+07,1.067478e+07,9.686552e+08,90.742439,3.327561
1,1,wind_onshore,1000000.0,2500.0,25.0,400.0,400000.0,1897.871866,31.161570,6.372797,...,6.372797e+06,0.0,0.0,1.883743e+07,7.523257e+07,4.394215e+07,1.067478e+07,9.602340e+08,89.953553,4.116447
2,2,wind_onshore,1000000.0,2500.0,25.0,400.0,400000.0,1385.621290,32.499593,6.895060,...,6.895060e+06,0.0,0.0,1.989490e+07,7.417510e+07,2.375541e+08,1.067478e+07,7.666221e+08,71.816222,22.253778
3,3,wind_onshore,1000000.0,2500.0,25.0,400.0,400000.0,1303.943342,31.048289,8.171383,...,8.171383e+06,0.0,0.0,2.059070e+07,7.347930e+07,2.627978e+08,1.067478e+07,7.413784e+08,69.451427,24.618573
4,4,wind_onshore,1000000.0,2500.0,25.0,400.0,400000.0,1486.841537,37.186059,7.746482,...,7.746482e+06,0.0,0.0,2.262091e+07,7.144909e+07,1.679665e+08,1.067478e+07,8.362097e+08,78.335106,15.734894


In [3]:
npv_million_eur = results["npv_eur"] / 1_000_000
lcoe_eur_per_mwh = results["lcoe_eur_per_mwh"]
levelized_net_margin_eur_per_mwh = results["levelized_net_margin_eur_per_mwh"]
summary = pd.concat(
    [
        npv_million_eur.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "NPV million EUR"
        ),
        lcoe_eur_per_mwh.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "LCOE EUR/MWh"
        ),
        levelized_net_margin_eur_per_mwh.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "Levelized net margin EUR/MWh"
        ),
    ],
    axis=1,
)

npv_signs = summarize_metric_signs(npv_million_eur)
npv_sign_summary = pd.DataFrame(
    {
        "NPV category": ["Non-negative (NPV >= 0)", "Negative (NPV < 0)"],
        "Simulation count": [
            npv_signs["non_negative_count"],
            npv_signs["negative_count"],
        ],
        "Simulation share": [
            npv_signs["non_negative_share"],
            1.0 - npv_signs["non_negative_share"],
        ],
    }
)

display(summary)
display(npv_sign_summary)


,NPV million EUR,LCOE EUR/MWh,Levelized net margin EUR/MWh
count,100000.000000,100000.000000,100000.000000
mean,145.654132,80.425299,13.644701
std,71.013206,6.652430,6.652430
min,-19.312437,65.354210,-1.809165
5%,34.866270,70.042872,3.266230
50%,145.876923,80.404428,13.665572
95%,256.484218,90.803770,24.027128
max,306.534636,95.879165,28.715790


,NPV category,Simulation count,Simulation share
0,Non-negative (NPV >= 0),99842,0.99842
1,Negative (NPV < 0),158,0.00158


In [4]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(npv_million_eur, bins=50, color="tab:gray", edgecolor="white", alpha=0.8)
ax.axvline(npv_million_eur.mean(), color="tab:blue", linewidth=2, label="Mean")
ax.axvline(npv_million_eur.median(), color="tab:orange", linewidth=2, label="Median")
ax.axvline(0, color="black", linewidth=1, linestyle="--", label="Break-even")

ax.set_title(f"Onshore wind NPV distribution (n={SAMPLE_SIZE})")
ax.set_xlabel("NPV (million EUR)")
ax.set_ylabel("Frequency")
ax.grid(axis="y", alpha=0.25)
ax.legend()

fig.tight_layout()
plt.show()

/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42973/563335227.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## levelized net margin histogram


In [5]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(
    levelized_net_margin_eur_per_mwh,
    bins=50,
    color="tab:green",
    edgecolor="white",
    alpha=0.8,
)
ax.axvline(
    levelized_net_margin_eur_per_mwh.mean(),
    color="tab:blue",
    linewidth=2,
    label="Mean",
)
ax.axvline(
    levelized_net_margin_eur_per_mwh.median(),
    color="tab:orange",
    linewidth=2,
    label="Median",
)
ax.axvline(0, color="black", linewidth=1, linestyle="--", label="Break-even")

ax.set_title(f"Onshore wind levelized net margin distribution (n={SAMPLE_SIZE})")
ax.set_xlabel("Levelized net margin (EUR/MWh)")
ax.set_ylabel("Frequency")
ax.grid(axis="y", alpha=0.25)
ax.legend()

fig.tight_layout()
plt.show()


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42973/844008486.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
annual_components = results[
    [
        "annual_revenue_eur",
        "annual_fixed_opex_eur",
        "annual_variable_opex_eur",
        "annual_fuel_cost_eur",
        "annual_emissions_cost_eur",
        "annual_net_cash_flow_eur",
    ]
] / 1_000_000

annual_components.mean().rename("Mean annual value, million EUR")

annual_revenue_eur           94.070000
annual_fixed_opex_eur        13.226627
annual_variable_opex_eur      7.230404
annual_fuel_cost_eur          0.000000
annual_emissions_cost_eur     0.000000
annual_net_cash_flow_eur     73.612970
Name: Mean annual value, million EUR, dtype: float64